In [ ]:
# Cell 1
import sys
sys.path.append('..')

import pandas as pd
from src.data.load_data import load_train, load_test
from src.features.build_features import build_master, get_feature_cols, align_test
from src.data.validation import assert_feature_frame
from src.utils import config

# Cell 2
flag_tr, acc_tr, enq_tr = load_train()
train = build_master(flag_tr, acc_tr, enq_tr)

flag_te, acc_te, enq_te = load_test()
test = build_master(flag_te, acc_te, enq_te)

feature_cols = get_feature_cols(train)
test = align_test(train, test, feature_cols)

print('Train:', train.shape)
print('Test :', test.shape)
print('Features:', len(feature_cols))

# Cell 3 — Guardrails
assert_feature_frame(train, feature_cols)
print('No NaN in train features:', not train[feature_cols].isna().any().any())
print('No NaN in test  features:', not test[feature_cols].isna().any().any())

# Cell 4 — Class separation
check_cols = [
    'pmt_max_dpd', 'pmt_n_90', 'pmt_ever_90plus', 'acc_n_with_overdue',
    'acc_open_ratio', 'acc_n_accounts', 'enq_n', 'enq_n_90d',
    'enq_days_since_last', 'is_cash_loan', 'has_accounts',
]
train.groupby('TARGET')[check_cols].mean().T

# Cell 5 — Correlations
corr = train[feature_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
print('TOP + correlated:\n', corr.sort_values(ascending=False).head(15))
print('\nTOP - correlated:\n', corr.sort_values().head(10))

# Cell 6 — Persist
config.ensure_dirs()
train.to_csv(config.FEATURES_TRAIN, index=False)
test.to_csv(config.FEATURES_TEST, index=False)
print('saved features to', config.PROCESSED_DIR)